# Informed Search and Heuristics

In this tutoring lecture we are gonna explore the difference between state-sapce graphs and search tree, and then we will put our hands on and try to develope some heuristics to solve a "simple" problem.

## State-Space Graphs vs. Search Trees

### State-Space Graphs
A **State-Space Graph** represents the physical world or the actual configuration space of the problem.
* Each unique state (e.g., state C) exists **only once**.
* Multiple distinct paths can lead to the same physical state (e.g., reaching state C via path $S \to A \to C$ or $S \to B \to C$).

### Search Trees
A **Search Tree** unfurls the graph choices over time, mapping out all possible action sequences starting from the initial state.
* If we use a pure **Tree Search**, identical states are treated as completely distinct nodes if reached via different pathways.
* This causes redundant expansions and an exponential blow-up in the number of nodes if the graph contains loops or converging paths.
* **Graph Search** solves this inefficiency by maintaining an `explored` set (closed list) to prune duplicate state evaluations.

![SSG vs ST](./imgs/ssg_vs_st.png)

## Heuristics

A **heuristic** $h(n)$ is a domain-specific function that estimates the cost of the cheapest path from a given state $n$ to the goal. It acts as an informed compass for search algorithms like A*.

### The Problem with UCS (Uniform Cost Search)
UCS expands nodes solely based on path cost $g(n)$. While it guarantees finding the optimal path, it explores blindly in all directions like an expanding oil spill, leading to massive node expansions as the search depth grows.

### Types of Heuristics & Relaxed Problems
We typically construct admissible heuristics by **relaxing** the problem constraints (making the rules easier):
1. **Admissible Heuristics:** A heuristic is admissible if it never overestimates the true cost to reach the goal ($h(n) \le h^*(n)$). This guarantees optimality in A* tree search.
2. **Consistent (Monotonic) Heuristics:** Satisfies the triangle inequality ($h(n) \le cost(n, n') + h(n')$), ensuring optimality in Graph Search without re-opening explored nodes.

## Hands On : Puzzle Time

### Logic & Difficulties of the 8-Puzzle
The **8-Puzzle** consists of a 3x3 grid with 8 numbered tiles and one empty space (represented by `0`)[cite: 2, 3]. Tiles can slide horizontally or vertically into the empty space.

**Difficulties in solving it:**
* The state space size is $9! / 2 = 181,440$ reachable configurations.
* Blind searches easily get trapped searching deep sub-trees without a sense of direction, resulting in thousands or millions of expanded nodes.

![Puzzle1](./imgs/puzzle1.png)
![Puzzle2](./imgs/puzzle2.png)

Here we have a simple implementation of the puzzle state. We will use this object here to solve the problem.

Just to give you clarity, and help you developing the class, here is a breakdown of what the state represent and what the methods should do.

* `__init__`: Initializes the board tuple, tracking its parent node, the action taken to reach it, path cost $g$, heuristic value $h$, and total evaluation score $f = g + h$.
* `state_key`: Returns a unique hashable representation (the board tuple) for the graph search explored set.
* `get_neighbors`: Locates the empty tile (`0`), calculates valid row/col movements (`UP`, `DOWN`, `LEFT`, `RIGHT`), and generates all legal successor states by swapping tiles.

In [ ]:
import heapq

class PuzzleState:
    def __init__(self, board, parent=None, action=None, cost=0, h=0):
        self.board = board       # Tuple representing 3x3 board, e.g., (7, 2, 4, 5, 0, 6, 8, 3, 1)
        self.parent = parent
        self.action = action
        self.g = cost            # Path cost (steps taken)
        self.h = h               # Heuristic value
        self.f = self.g + self.h

    @property
    def state_key(self):
        return self.board

    def get_neighbors(self):
        neighbors = []
        # Find index of the empty tile (0)
        idx = self.board.index(0)
        row, col = divmod(idx, 3)
        
        # Possible movements: Up, Down, Left, Right
        moves = {
            "UP": (-1, 0),
            "DOWN": (1, 0),
            "LEFT": (0, -1),
            "RIGHT": (0, 1)
        }
        
        for action, (dr, dc) in moves.items():
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_idx = new_row * 3 + new_col
                # Swap empty tile with the neighbor
                board_list = list(self.board)
                board_list[idx], board_list[new_idx] = board_list[new_idx], board_list[idx]
                neighbors.append((tuple(board_list), action))
                
        return neighbors

Time to code our heuristics! Remember:

1. **`h_zero` (UCS / $h = 0$):** No heuristic guidance[cite: 3]. Reduces A* search to Uniform Cost Search, causing heavy exploration.
2. **`h_misplaced_tiles` (Tiles / $h_1$):** Counts how many tiles are out of place compared to the goal (ignoring the empty tile). Based on the relaxed problem where any tile can jump to any position.
3. **`h_manhattan` (Manhattan Distance / $h_2$):** Sums the absolute horizontal and vertical grid distances each tile is from its target location. Based on the relaxed problem where tiles can move independently through one another.

In [ ]:
# --- Heuristic Functions ---
def h_zero(state, goal):
    """UCS: No heuristic guidance (h = 0)"""
    return 0

def h_misplaced_tiles(state, goal):
    """Tiles Heuristic: Number of tiles misplaced (excluding empty space 0)"""
    # TODO
    pass

def h_manhattan(state, goal):
    """Manhattan Distance Heuristic: Sum of grid distances for each tile"""
    # TODO
    pass

Let's then wrap everything up in order to write the final solver for the puzzle.\
The `solve_puzzle` function implements a priority queue (`heapq`) ordered by the evaluation score $f(n) = g(n) + h(n)$:
* It pops the lowest-cost node, checks if it matches the `goal_board`, and reconstructs the action path if successful.
* Depending on the `search_type` parameter, it toggles between **Graph Search** (using an `explored` set to filter out duplicate state evaluations) and **Tree Search**.

In [ ]:
def solve_puzzle(start_board, goal_board, heuristic_func, search_type="graph"):
    start_node = PuzzleState(
        board=start_board, 
        cost=0, 
        h=heuristic_func(start_board, goal_board)
    )
    
    # Priority queue ordered by f(n) = g(n) + h(n), breaking ties with node creation order
    frontier = []
    counter = 0  # tie-breaker for heap
    heapq.heappush(frontier, (start_node.f, counter, start_node))
    
    explored = set()
    nodes_expanded = 0
    max_frontier_size = 1

    while frontier:
        # TODO
        current = ...

        # Check goal
        if current.board == goal_board:
            # Reconstruct path
            path = []
            # TODO
            # here the format is :
                # success
                # path_length 
                # nodes_expanded
                # max_frontier
                # path 
            
        # Graph search checks explored set; Tree search does not!
        # TODO
            
        nodes_expanded += 1
        
        for neighbor_board, action in current.get_neighbors():
            # TODO
            # update the frontier
            ...
            
    return {"success": False, "nodes_expanded": nodes_expanded}

Here we have a simple animation function to visualize the solution found.

You could try to use dfs/bfs or some strange (non correct) heuristics and see what solution they found.

In [13]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
from IPython.display import HTML

def create_puzzle_gif(start_board, path, filename="puzzle_solution.gif"):
    """
    Creates and returns an animated GIF of the 8-puzzle solution path.
    """
    full_path = [(None, start_board)] + path
    
    fig, ax = plt.subplots(figsize=(4, 4))
    
    def update(frame_idx):
        ax.clear()
        ax.set_xticks([])
        ax.set_yticks([])
        
        action, board = full_path[frame_idx]
        grid = np.array(board).reshape(3, 3)
        
        for r in range(3):
            for c in range(3):
                val = grid[r, c]
                bg_color = "#333333" if val == 0 else "#e1f5fe"
                text_color = "white" if val == 0 else "#01579b"
                text_val = str(val) if val != 0 else ""
                
                ax.add_patch(plt.Rectangle((c, 2 - r), 1, 1, facecolor=bg_color, edgecolor="black", linewidth=1.5))
                if val != 0:
                    ax.text(c + 0.5, 2 - r + 0.5, text_val, fontsize=20, fontweight='bold', 
                            ha='center', va='center', color=text_color)
                    
        ax.set_xlim(0, 3)
        ax.set_ylim(0, 3)
        ax.set_aspect('equal')
        
        if frame_idx == 0:
            ax.set_title("Start State", fontsize=14, fontweight='bold', pad=10)
        else:
            ax.set_title(f"Step {frame_idx} | Action: {action}", fontsize=12, pad=10)

    anim = animation.FuncAnimation(fig, update, frames=len(full_path), interval=1000, repeat=True)
    
    anim.save(filename, writer='pillow', fps=1)
    plt.close(fig)

    # combine this with display()
    return HTML(f'<img src="{filename}" alt="Puzzle Animation">')

And here we have our final comparison.\
Feel free to change the starting state, the goal state, the heuristics etc. The imagination is your limit here.\
I also suggest you to do a little profiling adding some time counters, as we already did in previous lectures.

In [ ]:
start_state = (7, 2, 4, 
               5, 0, 6, 
               8, 3, 1)

goal_state  = (0, 1, 2, 
               3, 4, 5, 
               6, 7, 8)

res_ucs = solve_puzzle(start_state, goal_state, h_zero, search_type="graph")

res_tiles = solve_puzzle(start_state, goal_state, h_misplaced_tiles, search_type="graph")

res_manhattan = solve_puzzle(start_state, goal_state, h_manhattan, search_type="graph")

if res_manhattan["success"]:
    display(create_puzzle_gif(start_state, res_manhattan["path"]))
else:
    print("No solution found.")

print()

print(f"Strategy: UCS (Blind Search)\n    Nodes Expanded : {res_ucs["nodes_expanded"]}\n    Path Length : {res_ucs["path_length"]}")
print(f"Strategy: Misplaced Tiles (h1)\n    Nodes Expanded {res_tiles["nodes_expanded"]}\n    Path Length: {res_tiles["path_length"]}")
print(f"Strategy: Manhattan Distance (h2)\n    Nodes Expanded: {res_manhattan["nodes_expanded"]}\n    Path Length: {res_manhattan["path_length"]}")